# Rolex Watch Price Analysis - Part 5: Regression Modeling
## Final Project - Data Analytics

## Libraries and settings

In [ ]:
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import LabelEncoder

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print(os.getcwd())

## Import Cleaned Data

In [ ]:
df = pd.read_csv('rolex_data_cleaned.csv', encoding='utf-8')

print(f"Dataset shape: {df.shape}")
df.head()

## 1. Data Preparation for Modeling

### 1.1 Encode Categorical Variables

In [ ]:
df_model = df.copy()

le_condition = LabelEncoder()
le_material = LabelEncoder()
le_movement = LabelEncoder()

df_model['condition_encoded'] = le_condition.fit_transform(df_model['condition_category'])
df_model['material_encoded'] = le_material.fit_transform(df_model['material_category'])
df_model['movement_encoded'] = le_movement.fit_transform(df_model['movement_type'])

print("Encoding mappings:")
print("\nCondition:")
for i, label in enumerate(le_condition.classes_):
    print(f"  {i}: {label}")
print("\nMaterial:")
for i, label in enumerate(le_material.classes_):
    print(f"  {i}: {label}")
print("\nMovement:")
for i, label in enumerate(le_movement.classes_):
    print(f"  {i}: {label}")

### 1.2 Select Features for Modeling

In [ ]:
df_model = df_model.dropna(subset=['price', 'year', 'age'])

feature_cols = ['year', 'age', 'condition_encoded', 'material_encoded', 
                'movement_encoded', 'has_box', 'has_papers', 'has_complete_set', 'is_professional']

X = df_model[feature_cols]
y = df_model['price']

print(f"Feature matrix shape: {X.shape}")
print(f"Target variable shape: {y.shape}")
print(f"\nFeatures used for modeling:")
for i, col in enumerate(feature_cols, 1):
    print(f"  {i}. {col}")

### 1.3 Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print(f"Training set size: {X_train.shape[0]:,} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Test set size: {X_test.shape[0]:,} samples ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"\nTraining set price statistics:")
print(f"  Mean: {y_train.mean():,.2f} CHF")
print(f"  Median: {y_train.median():,.2f} CHF")
print(f"  Std: {y_train.std():,.2f} CHF")

## 2. Multiple Linear Regression (Statsmodels)

### 2.1 Fit the Model

In [ ]:
X_train_const = sm.add_constant(X_train)
X_test_const = sm.add_constant(X_test)

ols_model = sm.OLS(y_train, X_train_const)
ols_results = ols_model.fit()

print(ols_results.summary())

### 2.2 Model Interpretation

In [ ]:
print("Multiple Linear Regression - Key Results")
print("="*70)
print(f"\nR-squared: {ols_results.rsquared:.4f}")
print(f"Adjusted R-squared: {ols_results.rsquared_adj:.4f}")
print(f"F-statistic: {ols_results.fvalue:.2f}")
print(f"Prob (F-statistic): {ols_results.f_pvalue:.6f}")

print(f"\nCoefficients and Significance:")
coef_df = pd.DataFrame({
    'Feature': ['Intercept'] + feature_cols,
    'Coefficient': ols_results.params.values,
    'P-value': ols_results.pvalues.values,
    'Significant': ['***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else '' 
                    for p in ols_results.pvalues.values]
})
print(coef_df.to_string(index=False))

print(f"\nInterpretation:")
print(f"  - R-squared of {ols_results.rsquared:.4f} means the model explains {ols_results.rsquared*100:.2f}% of price variance")
print(f"  - Features with p < 0.05 have statistically significant effects on price")
print(f"  - Positive coefficients indicate price increases with the feature")
print(f"  - Negative coefficients indicate price decreases with the feature")

### 2.3 Model Predictions and Evaluation

In [ ]:
y_pred_train_ols = ols_results.predict(X_train_const)
y_pred_test_ols = ols_results.predict(X_test_const)

r2_train_ols = r2_score(y_train, y_pred_train_ols)
r2_test_ols = r2_score(y_test, y_pred_test_ols)
rmse_train_ols = np.sqrt(mean_squared_error(y_train, y_pred_train_ols))
rmse_test_ols = np.sqrt(mean_squared_error(y_test, y_pred_test_ols))
mae_train_ols = mean_absolute_error(y_train, y_pred_train_ols)
mae_test_ols = mean_absolute_error(y_test, y_pred_test_ols)

print("Multiple Linear Regression - Performance Metrics")
print("="*70)
print(f"\nTraining Set:")
print(f"  R-squared: {r2_train_ols:.4f}")
print(f"  RMSE: {rmse_train_ols:,.2f} CHF")
print(f"  MAE: {mae_train_ols:,.2f} CHF")
print(f"\nTest Set:")
print(f"  R-squared: {r2_test_ols:.4f}")
print(f"  RMSE: {rmse_test_ols:,.2f} CHF")
print(f"  MAE: {mae_test_ols:,.2f} CHF")

print(f"\nSample Predictions (first 10 test samples):")
comparison_df = pd.DataFrame({
    'Actual': y_test.values[:10],
    'Predicted': y_pred_test_ols[:10],
    'Difference': y_test.values[:10] - y_pred_test_ols[:10]
})
print(comparison_df.round(2).to_string())

### 2.4 Residuals Analysis

In [ ]:
residuals_train = y_train - y_pred_train_ols
residuals_test = y_test - y_pred_test_ols

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].scatter(y_pred_train_ols, residuals_train, s=10, alpha=0.5, color='#2E86AB')
axes[0, 0].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[0, 0].set_xlabel('Predicted Price (CHF)', fontsize=10)
axes[0, 0].set_ylabel('Residuals (CHF)', fontsize=10)
axes[0, 0].set_title('Residual Plot (Training Set)', fontsize=11, pad=10)
axes[0, 0].grid(alpha=0.3)

axes[0, 1].hist(residuals_train, bins=50, color='#2E86AB', edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Residuals (CHF)', fontsize=10)
axes[0, 1].set_ylabel('Frequency', fontsize=10)
axes[0, 1].set_title('Distribution of Residuals (Training)', fontsize=11, pad=10)
axes[0, 1].grid(axis='y', alpha=0.3)

axes[1, 0].scatter(y_test, y_pred_test_ols, s=10, alpha=0.5, color='#F18F01')
min_val = min(y_test.min(), y_pred_test_ols.min())
max_val = max(y_test.max(), y_pred_test_ols.max())
axes[1, 0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2)
axes[1, 0].set_xlabel('Actual Price (CHF)', fontsize=10)
axes[1, 0].set_ylabel('Predicted Price (CHF)', fontsize=10)
axes[1, 0].set_title(f'Actual vs Predicted (Test Set, R²={r2_test_ols:.4f})', fontsize=11, pad=10)
axes[1, 0].grid(alpha=0.3)

stats.probplot(residuals_train, dist="norm", plot=axes[1, 1])
axes[1, 1].set_title('Q-Q Plot of Residuals', fontsize=11, pad=10)
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Residuals Analysis:")
print(f"  Mean of residuals: {residuals_train.mean():.2f} CHF (should be close to 0)")
print(f"  Std of residuals: {residuals_train.std():,.2f} CHF")

### Interpretation

Residual analysis helps assess model assumptions:
- Residuals should be randomly scattered around zero (no patterns)
- Histogram should approximate a normal distribution
- Q-Q plot should show points close to the diagonal line
- Actual vs Predicted plot shows how well predictions match reality

## 3. Random Forest Regression

### 3.1 Fit the Random Forest Model

In [ ]:
rf_model = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

print("Random Forest Model Parameters:")
print(f"  Number of trees: {rf_model.n_estimators}")
print(f"  Max depth: {rf_model.max_depth}")
print(f"  Number of features: {rf_model.n_features_in_}")
print(f"\nModel training completed successfully.")

### 3.2 Model Predictions and Evaluation

In [ ]:
y_pred_train_rf = rf_model.predict(X_train)
y_pred_test_rf = rf_model.predict(X_test)

r2_train_rf = r2_score(y_train, y_pred_train_rf)
r2_test_rf = r2_score(y_test, y_pred_test_rf)
rmse_train_rf = np.sqrt(mean_squared_error(y_train, y_pred_train_rf))
rmse_test_rf = np.sqrt(mean_squared_error(y_test, y_pred_test_rf))
mae_train_rf = mean_absolute_error(y_train, y_pred_train_rf)
mae_test_rf = mean_absolute_error(y_test, y_pred_test_rf)

print("Random Forest Regression - Performance Metrics")
print("="*70)
print(f"\nTraining Set:")
print(f"  R-squared: {r2_train_rf:.4f}")
print(f"  RMSE: {rmse_train_rf:,.2f} CHF")
print(f"  MAE: {mae_train_rf:,.2f} CHF")
print(f"\nTest Set:")
print(f"  R-squared: {r2_test_rf:.4f}")
print(f"  RMSE: {rmse_test_rf:,.2f} CHF")
print(f"  MAE: {mae_test_rf:,.2f} CHF")

print(f"\nSample Predictions (first 10 test samples):")
comparison_df_rf = pd.DataFrame({
    'Actual': y_test.values[:10],
    'Predicted': y_pred_test_rf[:10],
    'Difference': y_test.values[:10] - y_pred_test_rf[:10]
})
print(comparison_df_rf.round(2).to_string())

### 3.3 Feature Importance

In [ ]:
feature_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(range(len(feature_importance)), feature_importance['Importance'].values, 
         color='#2E86AB', alpha=0.8)
plt.yticks(range(len(feature_importance)), feature_importance['Feature'].values)
plt.xlabel('Importance', fontsize=11)
plt.ylabel('Feature', fontsize=11)
plt.title('Random Forest Feature Importance', fontsize=12, pad=10)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("Feature Importance Ranking:")
for i, row in feature_importance.iterrows():
    print(f"  {row['Feature']:25s}: {row['Importance']:.4f}")

### Interpretation

Feature importance in Random Forest indicates which variables contribute most to prediction accuracy. Higher values mean the feature is more important for making accurate predictions.

### 3.4 Random Forest Residuals Analysis

In [ ]:
residuals_train_rf = y_train - y_pred_train_rf
residuals_test_rf = y_test - y_pred_test_rf

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_pred_test_rf, residuals_test_rf, s=10, alpha=0.5, color='#F18F01')
axes[0].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[0].set_xlabel('Predicted Price (CHF)', fontsize=10)
axes[0].set_ylabel('Residuals (CHF)', fontsize=10)
axes[0].set_title('Residual Plot - Random Forest (Test Set)', fontsize=11, pad=10)
axes[0].grid(alpha=0.3)

axes[1].scatter(y_test, y_pred_test_rf, s=10, alpha=0.5, color='#F18F01')
min_val = min(y_test.min(), y_pred_test_rf.min())
max_val = max(y_test.max(), y_pred_test_rf.max())
axes[1].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2)
axes[1].set_xlabel('Actual Price (CHF)', fontsize=10)
axes[1].set_ylabel('Predicted Price (CHF)', fontsize=10)
axes[1].set_title(f'Actual vs Predicted - RF (Test Set, R²={r2_test_rf:.4f})', fontsize=11, pad=10)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Random Forest Residuals Statistics (Test Set):")
print(f"  Mean of residuals: {residuals_test_rf.mean():.2f} CHF")
print(f"  Std of residuals: {residuals_test_rf.std():,.2f} CHF")
print(f"  Min residual: {residuals_test_rf.min():,.2f} CHF")
print(f"  Max residual: {residuals_test_rf.max():,.2f} CHF")

## 4. Model Comparison

In [ ]:
comparison_df = pd.DataFrame({
    'Model': ['Linear Regression', 'Linear Regression', 'Random Forest', 'Random Forest'],
    'Dataset': ['Training', 'Test', 'Training', 'Test'],
    'R-squared': [r2_train_ols, r2_test_ols, r2_train_rf, r2_test_rf],
    'RMSE': [rmse_train_ols, rmse_test_ols, rmse_train_rf, rmse_test_rf],
    'MAE': [mae_train_ols, mae_test_ols, mae_train_rf, mae_test_rf]
})

print("Model Performance Comparison")
print("="*80)
print(comparison_df.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

models = ['Linear\nRegression\n(Train)', 'Linear\nRegression\n(Test)', 
          'Random\nForest\n(Train)', 'Random\nForest\n(Test)']
r2_values = [r2_train_ols, r2_test_ols, r2_train_rf, r2_test_rf]
rmse_values = [rmse_train_ols, rmse_test_ols, rmse_train_rf, rmse_test_rf]
mae_values = [mae_train_ols, mae_test_ols, mae_train_rf, mae_test_rf]
colors = ['#2E86AB', '#5BA3C7', '#F18F01', '#FFA938']

axes[0].bar(range(4), r2_values, color=colors, alpha=0.8)
axes[0].set_xticks(range(4))
axes[0].set_xticklabels(models, fontsize=9)
axes[0].set_ylabel('R-squared', fontsize=11)
axes[0].set_title('R-squared Comparison', fontsize=12, pad=10)
axes[0].set_ylim([0, 1])
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(range(4), rmse_values, color=colors, alpha=0.8)
axes[1].set_xticks(range(4))
axes[1].set_xticklabels(models, fontsize=9)
axes[1].set_ylabel('RMSE (CHF)', fontsize=11)
axes[1].set_title('RMSE Comparison', fontsize=12, pad=10)
axes[1].grid(axis='y', alpha=0.3)

axes[2].bar(range(4), mae_values, color=colors, alpha=0.8)
axes[2].set_xticks(range(4))
axes[2].set_xticklabels(models, fontsize=9)
axes[2].set_ylabel('MAE (CHF)', fontsize=11)
axes[2].set_title('MAE Comparison', fontsize=12, pad=10)
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nInterpretation:")
print(f"  - R-squared: Higher is better (closer to 1 means better fit)")
print(f"  - RMSE & MAE: Lower is better (smaller prediction errors)")
if r2_test_rf > r2_test_ols:
    print(f"  - Random Forest performs better on test data (R² = {r2_test_rf:.4f} vs {r2_test_ols:.4f})")
else:
    print(f"  - Linear Regression performs better on test data (R² = {r2_test_ols:.4f} vs {r2_test_rf:.4f})")
    
overfitting_ols = r2_train_ols - r2_test_ols
overfitting_rf = r2_train_rf - r2_test_rf
print(f"  - Linear Regression overfitting gap: {overfitting_ols:.4f}")
print(f"  - Random Forest overfitting gap: {overfitting_rf:.4f}")

## 5. Predictions Comparison on Sample Data

In [ ]:
sample_size = 20
sample_indices = np.random.choice(len(y_test), sample_size, replace=False)

comparison_sample = pd.DataFrame({
    'Actual': y_test.values[sample_indices],
    'Linear_Reg': y_pred_test_ols[sample_indices],
    'Random_Forest': y_pred_test_rf[sample_indices],
    'Error_LR': y_test.values[sample_indices] - y_pred_test_ols[sample_indices],
    'Error_RF': y_test.values[sample_indices] - y_pred_test_rf[sample_indices]
})

print(f"Sample Predictions Comparison (random {sample_size} test samples):")
print("="*80)
print(comparison_sample.round(2).to_string())

x_pos = np.arange(sample_size)
width = 0.25

plt.figure(figsize=(14, 6))
plt.bar(x_pos - width, comparison_sample['Actual'], width, label='Actual', color='green', alpha=0.7)
plt.bar(x_pos, comparison_sample['Linear_Reg'], width, label='Linear Regression', color='#2E86AB', alpha=0.7)
plt.bar(x_pos + width, comparison_sample['Random_Forest'], width, label='Random Forest', color='#F18F01', alpha=0.7)
plt.xlabel('Sample Index', fontsize=11)
plt.ylabel('Price (CHF)', fontsize=11)
plt.title('Predictions Comparison: Actual vs Linear Regression vs Random Forest', fontsize=12, pad=10)
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Jupyter notebook --footer info-- (please always provide this at the end of each submitted notebook)

In [ ]:
import os
import platform
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')